In [1]:
import os
import numpy as np
import matplotlib.pyplot as plt
from astropy.io import fits
from astropy.wcs import WCS
from astropy.table import Table
from gliders.utils import sigma_clipping, grow_mask, get_profile_from_border, normalize_data
from gliders.artifacts import detect_artifacts, get_artifact_flag, clean_artifacts, detect_and_clean_artifacts
from gliders.background_suppressor import suppress_background, show_all_methods
from scipy.stats import norm
from scipy.signal import convolve2d
from astropy.convolution import Gaussian2DKernel
import warnings
from astropy.utils.exceptions import AstropyDeprecationWarning
from photutils.utils.exceptions import NoDetectionsWarning
warnings.simplefilter('ignore', category=AstropyDeprecationWarning)
warnings.simplefilter('ignore', category=NoDetectionsWarning)

def extract_band_data(hdu_list, verbose=True, bands_to_analyze = ['r']):
    all_bands = {}
    all_img_diffs = {}
    all_img_areas = {}
    all_weights = {}
    all_wgh_diffs = {}
    all_wgh_areas = {}
    ref_hdu = None
    for hdu in hdu_list:
        try:
            header = hdu._header
            if verbose:
                print(header)
            band = header.get("BAND")
            if band not in bands_to_analyze: continue
            image_type = header.get("IMAGETYP")
            imagew = header.get("IMAGEW")
            imageh = header.get("IMAGEH")

            if band is None or image_type is None or imagew is None or imageh is None:
                continue  # Skip if essential header info is missing

            img_dim_diff = abs(imagew - imageh)
            img_area = imagew * imageh

            if image_type == "image":
                if band not in all_bands or img_area > all_img_areas[band] or (img_area == all_img_areas[band] and img_dim_diff < all_img_diffs[band]):
                    all_bands[band] = hdu.copy().data
                    all_img_areas[band] = img_area
                    all_img_diffs[band] = img_dim_diff
                    ref_hdu = hdu.header

            elif image_type == "invvar":
                w_map = hdu.copy().data
                w_map[w_map == 0] += 1  # Avoid division by zero
                weight = 1 / np.sqrt(w_map)
                if band not in all_weights or img_area > all_wgh_areas[band] or (img_area == all_wgh_areas[band] and img_dim_diff < all_wgh_diffs[band]):
                    all_weights[band] = weight
                    all_wgh_areas[band] = img_area
                    all_wgh_diffs[band] = img_dim_diff

        except Exception as e:
            print(f"Skipped HDU due to error: {e}")

    return all_bands, all_weights, ref_hdu

file_dir = "C:/Astrostuff/PhD/Data/DESI FITS/"
artif_file = "D:/Astrostuff/PhD/Data/My_data/DESI_New_mergers.csv"
artif_catalog = Table.read(artif_file, format = "csv", converters = {"object_id":str})
print(len(artif_catalog))
print(artif_catalog)

3754
    object_id         ra      dec       z    ...    I    merger_predicted  ES
----------------- --------- -------- ------- ... ------- ---------------- ---
39627902957912363 211.09828  4.71795 0.02977 ... 0.41312                1   1
39632991730798910 253.11914 35.41754 0.05766 ...  0.2024                1   1
39627829528234714 150.37899  1.68886 0.09869 ... 0.01352                1   1
39627830664894213  218.2308  1.86938 0.02998 ... 0.58443                1   1
39633427988745611 271.28566 63.42304  0.0915 ...  0.2459                1   1
39627758090851335  212.4838 -1.23884  0.0248 ... 0.47045                1   1
39633104083616306 152.75604 41.58966 0.05861 ... 0.60419                1   1
39627877821449269 150.00236  3.67734 0.07391 ... 0.03693                1   1
39627773882401920   73.5449 -0.39993 0.05467 ... 0.15411                1   1
39633321931571417 185.66454 55.08292 0.03664 ... 0.28208                1   1
              ...       ...      ...     ... ...     ...   

In [4]:
bleed_radii = []
areas = []
s_detections = {'fwhm':[], 'kernel_std':[], 'detections':[], 'source':[]}
from gliders.cleaning_configs import configs
import pandas as pd
config = configs["default"]
for ii, object_info in enumerate(artif_catalog[140:]):
    if ii <= 0:
        continue
    object_id = object_info[0]
    # print(ii, object_id)
    obj_coords = (object_info[1],object_info[2])
    redshift = object_info[3]
    bands = ['r'] #,'g']
    file_name = f"{object_id}_{"".join(bands)}.fits"
    sources_name = f"{object_id}_sources.fits"

    file_path = os.path.join(file_dir, file_name)
    sources_path = os.path.join(file_dir, sources_name)
    try:
        with fits.open(file_path) as hdu_list:
            # hdu_list.info()
            all_bands, all_weights, ref_header = extract_band_data(hdu_list, verbose=False, bands_to_analyze=bands)
    except:
        continue

    with fits.open(sources_path) as hdul:
        data = hdul[1].data
        as_table = Table(data)
    wcs = WCS(ref_header)
    tractor_x_pix, tractor_y_pix = wcs.all_world2pix(as_table['ra'], as_table['dec'], 0)  # origin=0 for Python indexing
    source_x_pix, source_y_pix = wcs.all_world2pix(obj_coords[0], obj_coords[1], 0)  # origin=0 for Python indexing
    source_pos = (np.round(source_x_pix), np.round(source_y_pix))
    as_table['x_centroid'] = tractor_x_pix
    as_table['y_centroid'] = tractor_y_pix
    phot_sources = as_table['type','x_centroid','y_centroid']
    image = all_bands['r']


    if obj_coords[1] > 32.375:
        pix_scale = 1.72
    else:
        pix_scale = 1

    from photutils.segmentation import SourceCatalog, detect_threshold, detect_sources
    # from gliders.background_suppressor import suppress_background
    from gliders.stars import detect_and_mask_stars, find_all_star_apertures, plot_apertures, classify_apertures, detect_stars_in_image, get_star_apertures
    from astropy.table import vstack, hstack
    source_min_pix = 10
    # image = suppress_background(image, method = "layers")
    from gliders.pipeline import update_detections
    from gliders.utils import convolve_with_gaussian_kernel, smooth_blend
    from photutils.segmentation import SegmentationImage
    from astropy.stats import SigmaClip

    # image = suppress_background(image, method = "layers")
    image_u, seg_map, main_source_segmap = update_detections(image,source_min_pix=20, source_pos=source_pos, use_filter=True)

    main_source_mask = main_source_segmap.make_source_mask()
    main_source_npixels = np.sum(main_source_mask)

    # safe_star_masking_radius = round(np.sqrt(main_source_npixels * config.safe_star_masking_prop_radius / np.pi))
    # print(safe_star_masking_radius)



    # safe_star_masking_radius = 0
    # all_star_apertures = find_all_star_apertures(image, source_pos, phot_sources, safe_star_masking_radius, config, use_phot_sources = True, use_finders = False, clear_nucleus = False, image_psf = None)

    all_stars_detected_list = []
    convolved_image = image
    # detector = config.star_detector
    detector = "dao"
    main_fwhm = 5
    them_stds = np.linspace(1,3,21)
    them_fwhm = np.linspace(1,53,14)
    # print(them_stds)
    # print(them_fwhm)
    # break
    # them_stds = np.linspace(1,5,6)
    # them_fwhm = np.linspace(1,30,5)
    # print(config.star_psf_stds)
    # for this_std in config.star_psf_stds:
    for this_std in them_stds:
        if this_std > 0:
            # Instead of convolving the original image with a psf with this_std we use the previous convoluted image and convolve it with effective_std, which is equivalent but results in half processing time.
            effective_std = np.sqrt(this_std*this_std-np.power(this_std-1,2))
            # print(effective_std)
            convolved_image = convolve_with_gaussian_kernel(convolved_image, effective_std)

        # main_fwhm = 8.6 + 3.5 * this_std

        # this_fwhm = main_fwhm
        # print(this_std, main_fwhm)
        # for this_fwhm in [main_fwhm - fwhm_span, main_fwhm, main_fwhm + fwhm_span]:
        for this_fwhm in them_fwhm:

            stars_detected = detect_stars_in_image(convolved_image, config.star_detection_threshold,
                                                   detector, this_fwhm, config.star_roundness,
                                                   config.star_sharpness_low, config.star_min_radius, config.sigma_radius)
        # if stars_detected is not None:
        #     # Controlling that centroids are inside the image
        #     stars_detected['x_centroid'][stars_detected['x_centroid'] > image.shape[1] - 1] = image.shape[1] - 1
        #     stars_detected['x_centroid'][stars_detected['x_centroid'] < 0] = 0
        #     stars_detected['y_centroid'][stars_detected['y_centroid'] > image.shape[0] - 1] = image.shape[0] - 1
        #     stars_detected['y_centroid'][stars_detected['y_centroid'] < 0] = 0
        #     stars_detected['type'] = 'PSF'
        #     all_stars_detected_list.append(stars_detected)
            detections = 0
            if stars_detected is not None:
                detections = len(stars_detected)
            s_detections['fwhm'].append(round(this_fwhm,1))
            s_detections['kernel_std'].append(round(this_std,1))
            s_detections['detections'].append(detections)
            s_detections['source'].append(object_id)
            # print(this_fwhm, this_std, detections, object_id)



    # print(df)
    # break
    # print(all_stars_detected_list)

    # if len(all_stars_detected_list) > 0:
    #     all_stars_detected = vstack(all_stars_detected_list, metadata_conflicts = 'silent')
    #     all_stars_detected['id'] = np.arange(1, len(all_stars_detected) + 1)
    #     all_star_apertures = get_star_apertures(all_stars_detected)
    #
    #
    #     all_star_apertures_split = classify_apertures(all_star_apertures, main_source_mask)
    #     plot_apertures(image, all_star_apertures_split)
    # print(all_star_apertures)

    # plt.imshow(main_source_mask)
    # plt.show()

    # if config.auto_safe_masking_region:
    #     safe_star_masking_radius = round(np.sqrt(main_source_npixels * config.safe_star_masking_prop_radius / np.pi))
    #     if safe_star_masking_radius < 1: safe_star_masking_radius = 1
    # else:
    #     safe_star_masking_radius = config.safe_star_masking_radius
    #
    # all_star_apertures = find_all_star_apertures(image, main_source_pos, phot_sources, safe_star_masking_radius, config, use_phot_sources = True, use_finders = False, clear_nucleus = True, image_psf = image_psf)
    print(ii)
    if ii >= 30:
        break
df = pd.DataFrame(s_detections)
# bleed_radii = np.array(bleed_radii)
# areas = np.array(areas)
df.to_csv("C:\\Astrostuff\\PhD\\Data\\star_detections.csv", index = False)
print("DONE!")

2
4
5
6
7
8
9
11
12
15
16
17
20
21
22
23
26
29
30
DONE!
